## โจทย์
นักวิจัยทางชีววิทยาต้องการศึกษาความคล้ายคลึงกันทางกายภาพของนกเพนกวินในหมู่เกาะ Palmer Archipelago โดยต้องการจัดกลุ่ม (Clustering) นกเพนกวินจากลักษณะทางกายภาพ **โดยไม่ต้องพึ่งพาข้อมูลสายพันธุ์ (Species) ที่มีอยู่** เพื่อดูว่าอัลกอริทึมสามารถแบ่งกลุ่มได้ใกล้เคียงกับธรรมชาติหรือไม่

*ที่มาของ Dataset: https://www.kaggle.com/code/parulpandey/penguin-dataset-the-new-iris/notebook*

## Dataset: `penguins.csv`
ข้อมูลของนกเพนกวิน ประกอบด้วยคอลัมน์ดังนี้:
* `species`: สายพันธุ์
* `island`: ชื่อเกาะ
* `culmen_length_mm`: ความยาวจงอยปาก (มิลลิเมตร)
* `culmen_depth_mm`: ความลึกจงอยปาก (มิลลิเมตร)
* `flipper_length_mm`: ความยาวครีบ (มิลลิเมตร)
* `body_mass_g`: น้ำหนักตัว (กรัม)
* `sex`: เพศของนกเพนกวิน

## สิ่งที่ต้องทำ
1. **Data Preprocessing**
2. **Modeling & Clustering**
3. **Evaluation**
4. **สรุปผล**: สรุปว่าโมเดลใดและพารามิเตอร์ใดให้ผลลัพธ์ที่ดีที่สุด



---
### Import Library



In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle as pk
import json
import requests

# สำหรับ Data Preprocessing
from sklearn.preprocessing import StandardScaler

# สำหรับ Clustering Models
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans

# สำหรับ Deep Learning (Autoencoder)
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense

# สำหรับ Save .pkl
from sklearn.neighbors import KNeighborsClassifier

# สำหรับ Evaluation Metrics
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# สำหรับ Deployment (Flask)
from flask import Flask, request, jsonify

# ปิด Warning เพื่อให้ Output อ่านง่าย (ไม่จำเป็น)
import warnings
warnings.filterwarnings('ignore')

---
### 1. Data Preprocessing
* โหลดข้อมูล penguins.csv และสำรวจข้อมูลเบื้องต้น
* จัดการ Missing Values ด้วยวิธีที่เหมาะสม
* พิจารณาและเลือก Features ที่คิดว่ามีความจำเป็นและเหมาะสมต่อการทำ Clustering
* ทำ Data Transformation / Scaling ข้อมูลตามความจำเป็นของอัลกอริทึมที่จะเลือกใช้
* Clustering คือการเรียนรู้แบบไม่มีเฉลย (Unsupervised Learning) ไม่ต้อง (และ ไม่ควร) เปลี่ยนข้อมูลสายพันธุ์ (Species) เป็นตัวเลขเพื่อใส่เข้าไปในโมเดล

In [2]:
# 1. โหลดข้อมูล
df = pd.read_csv('penguins.csv')
df.head()

# 2. สำรวจและจัดการ Missing Values (Drop แถวที่มีค่าว่าง)
df_clean = df.dropna().copy()

# 3. เลือกเฉพาะ Features ตัวเลขสำหรับการทำ Clustering
features = ['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g']
X = df_clean[features]

# 4. Data Transformation / Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("ข้อมูลพร้อมสำหรับการ Train:")
print(f"ขนาดของข้อมูล X_scaled: {X_scaled.shape}")

ข้อมูลพร้อมสำหรับการ Train:
ขนาดของข้อมูล X_scaled: (334, 4)


In [3]:
df['species'].value_counts()
# df.head()

species
Adelie Penguin (Pygoscelis adeliae)          152
Gentoo penguin (Pygoscelis papua)            124
Chinstrap penguin (Pygoscelis antarctica)     68
Name: count, dtype: int64



---
### 2. Modeling & Clustering
* เลือกใช้อัลกอริทึม Clustering อย่างน้อย 3 โมเดล ที่คิดว่าเหมาะสมกับข้อมูลชุดนี้ โดยมีอย่างน้อย 1 โมเดลที่เป็น Deep Learning


In [4]:
# --- โมเดลที่ 1: Agglomerative Hierarchical Clustering ---
# ทราบจากข้อมูลธรรมชาติว่ามี 3 สายพันธุ์ จึงกำหนด n_clusters=3
agglo = AgglomerativeClustering(n_clusters=3)
agglo_labels = agglo.fit_predict(X_scaled)

# --- โมเดลที่ 2: DBSCAN (Density-based) ---
# ต้องปรับจูน eps และ min_samples ไม่เช่นนั้นอาจจัดเป็นกลุ่มเดียวหรือ Noise ทั้งหมด
dbscan = DBSCAN(eps=1.2, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_scaled)

# --- โมเดลที่ 3: Deep Learning (Autoencoder + Clustering) ---
# เนื่องจาก
# ประยุกต์โครงสร้าง Sequential Neural Network ให้อยู่ในรูป Autoencoder
input_dim = X_scaled.shape[1]

# Create a neural network model with hidden layers with 64 units and ReLU activation
autoencoder = Sequential([
    Dense(64, activation='relu', input_shape=(input_dim,)), # Index 0: Hidden layer 1
    Dense(2, activation='relu'),                            # Index 1: Latent space (บีบอัดฟีเจอร์)
    Dense(64, activation='relu'),                           # Index 2: Hidden layer 2
    Dense(input_dim, activation='linear')                   # Index 3: Output layer
])

# Compile the model with adam optimizer and mean squared error loss function
autoencoder.compile(optimizer='adam', loss='mse')

# Fit the model with 50 epochs and batch size of 16
print("Training Autoencoder...")
history = autoencoder.fit(X_scaled, X_scaled, epochs=50, batch_size=16, verbose=0)

# Predict (สกัดฟีเจอร์)
# ดึงข้อมูลจากเลเยอร์ที่ 2 (Index 1) มาใช้เพื่อลดมิติข้อมูล
# Use autoencoder.layers[0].input to explicitly get the input tensor
encoder = Model(inputs=autoencoder.layers[0].input, outputs=autoencoder.layers[1].output)
X_encoded = encoder.predict(X_scaled, verbose=0)

# นำฟีเจอร์ที่ผ่าน DL มาจัดกลุ่มต่อด้วย KMeans ให้เป็น 3 กลุ่ม
dl_cluster = KMeans(n_clusters=3, random_state=42)
dl_labels = dl_cluster.fit_predict(X_encoded)

print("\nทำ Modeling ทั้ง 3 อัลกอริทึมเสร็จสิ้น!")

Training Autoencoder...

ทำ Modeling ทั้ง 3 อัลกอริทึมเสร็จสิ้น!




---
### 3. Evaluation
* เลือกใช้ Evaluation Metrics ที่เหมาะสม 3 ตัวชี้วัด
* เขียนโค้ดเพื่อคำนวณและแสดงผลคะแนนของทุกโมเดลที่เลือกใช้


In [5]:
# ฟังก์ชันสำหรับคำนวณและแสดงผล Metrics
def evaluate_clustering(model_name, X_data, labels):
    # ตรวจสอบว่าโมเดลแยกกลุ่มได้มากกว่า 1 กลุ่มหรือไม่ (และไม่ใช่มีแต่ Noise -1)
    unique_labels = np.unique(labels)
    n_clusters = len(unique_labels) - (1 if -1 in unique_labels else 0)

    if n_clusters > 1:
        sil = silhouette_score(X_data, labels)
        cal = calinski_harabasz_score(X_data, labels)
        dav = davies_bouldin_score(X_data, labels)

        print(f"===== {model_name} =====")
        print(f"Silhouette Score (ยิ่งใกล้ 1 ยิ่งดี)   : {sil:.4f}")
        print(f"Calinski-Harabasz Score (ยิ่งมากยิ่งดี) : {cal:.4f}")
        print(f"Davies-Bouldin Score (ยิ่งน้อยยิ่งดี)  : {dav:.4f}\n")
    else:
        print(f"===== {model_name} =====")
        print(f"ไม่สามารถคำนวณ Metrics ได้เนื่องจากโมเดลพบเพียง {n_clusters} กลุ่ม\n")

# --- ประเมินผลโมเดลที่ 1: Agglomerative Clustering ---
evaluate_clustering("Agglomerative Clustering", X_scaled, agglo_labels)

# --- ประเมินผลโมเดลที่ 2: DBSCAN ---
evaluate_clustering("DBSCAN", X_scaled, dbscan_labels)

# --- ประเมินผลโมเดลที่ 3: Deep Learning (Autoencoder + KMeans) ---
# หมายเหตุ: ใช้ X_scaled เป็นตัวอ้างอิงในการวัดผลเพื่อให้เทียบกับโมเดลอื่นได้มาตรฐานเดียวกัน
evaluate_clustering("Deep Learning (Autoencoder + KMeans)", X_scaled, dl_labels)

===== Agglomerative Clustering =====
Silhouette Score (ยิ่งใกล้ 1 ยิ่งดี)   : 0.4527
Calinski-Harabasz Score (ยิ่งมากยิ่งดี) : 409.7105
Davies-Bouldin Score (ยิ่งน้อยยิ่งดี)  : 0.8497

===== DBSCAN =====
Silhouette Score (ยิ่งใกล้ 1 ยิ่งดี)   : 0.4673
Calinski-Harabasz Score (ยิ่งมากยิ่งดี) : 240.3768
Davies-Bouldin Score (ยิ่งน้อยยิ่งดี)  : 0.6148

===== Deep Learning (Autoencoder + KMeans) =====
Silhouette Score (ยิ่งใกล้ 1 ยิ่งดี)   : 0.2940
Calinski-Harabasz Score (ยิ่งมากยิ่งดี) : 214.2590
Davies-Bouldin Score (ยิ่งน้อยยิ่งดี)  : 1.2240





---
### 4. สรุปผลและบันทึก Model ที่เลือกเป็นไฟล์ `.pkl`
* เปรียบเทียบประสิทธิภาพของแต่ละโมเดลจาก Metrics ที่วัดได้
* สรุปว่าโมเดลใดเหมาะสมที่สุดสำหรับข้อมูลชุดนี้ พร้อมอธิบายเหตุผลสั้นๆ ประกอบการตัดสินใจ
* บันทึกเป็นไฟล์ .pkl


In [6]:
# ==========================================
# 4. สรุปผลและบันทึก Model สำหรับ Deployment
# ==========================================

print("""
[สรุปผลการทดลอง]
จากการเปรียบเทียบ Metrics ทั้ง 3 โมเดล พบว่า:
1. Deep Learning (Autoencoder) ให้ผลลัพธ์ที่แย่ที่สุด เนื่องจากข้อมูลมีเพียง 4 มิติ การใช้ Neural Network เข้ามาบีบอัดข้อมูลจึงเป็น Overkill และทำให้สูญเสียสาระสำคัญของข้อมูลไป
2. แม้ DBSCAN จะจัดการความหนาแน่นได้ดี แต่ 'Agglomerative Clustering' มีความโดดเด่นอย่างมากที่ค่า Calinski-Harabasz Score (409.71) ซึ่งสูงที่สุด สะท้อนให้เห็นว่ากลุ่มที่ถูกแบ่งมีความชัดเจน หนาแน่น และแยกออกจากกันได้ดีที่สุด

ดังนั้นจึงเลือกโมเดล Agglomerative Clustering เป็นโมเดลที่ดีที่สุดสำหรับข้อมูลชุดนี้
""")


# ---------------------------------------------------------
# 🔥 ทริคระดับโปร: การนำ Agglomerative ไปทำ API (Deployment)
# ---------------------------------------------------------
# ปัญหา: Agglomerative Clustering ไม่มีคำสั่ง model.predict() สำหรับรับข้อมูลใหม่ในอนาคต
# วิธีแก้: เราจะนำ Label ที่จัดกลุ่มได้สมบูรณ์แล้ว มาสอนให้ KNN (K-Nearest Neighbors)
# จดจำแพทเทิร์นนั้นไว้ เพื่อให้ KNN ทำหน้าที่เป็นตัว Predict ข้อมูลใหม่แทนในฝั่ง API

# 1. สร้างและเทรนโมเดล KNN เพื่อทำหน้าที่แทน (Proxy Model)
proxy_model = KNeighborsClassifier(n_neighbors=3)
proxy_model.fit(X_scaled, agglo_labels) # สอนด้วย X ที่ถูก scale และเฉลยจาก Agglomerative

# 2. บันทึกโมเดล KNN (ตัวแทน) และ Scaler ลงไฟล์ .pkl
deploy_data = {
    'model': proxy_model, # ใช้ KNN ที่เรียนรู้กลุ่มมาแล้วเป็นตัวทำนาย
    'scaler': scaler
}

with open('best_penguin_model.pkl', 'wb') as file:
    pk.dump(deploy_data, file)

print("\n✅ สร้าง Proxy Model และบันทึกไฟล์ 'best_penguin_model.pkl' สำเร็จ!")


[สรุปผลการทดลอง]
จากการเปรียบเทียบ Metrics ทั้ง 3 โมเดล พบว่า:
1. Deep Learning (Autoencoder) ให้ผลลัพธ์ที่แย่ที่สุด เนื่องจากข้อมูลมีเพียง 4 มิติ การใช้ Neural Network เข้ามาบีบอัดข้อมูลจึงเป็น Overkill และทำให้สูญเสียสาระสำคัญของข้อมูลไป
2. แม้ DBSCAN จะจัดการความหนาแน่นได้ดี แต่ 'Agglomerative Clustering' มีความโดดเด่นอย่างมากที่ค่า Calinski-Harabasz Score (409.71) ซึ่งสูงที่สุด สะท้อนให้เห็นว่ากลุ่มที่ถูกแบ่งมีความชัดเจน หนาแน่น และแยกออกจากกันได้ดีที่สุด

ดังนั้นจึงเลือกโมเดล Agglomerative Clustering เป็นโมเดลที่ดีที่สุดสำหรับข้อมูลชุดนี้


✅ สร้าง Proxy Model และบันทึกไฟล์ 'best_penguin_model.pkl' สำเร็จ!




---
### 5. Model Deployment (REST API with Flask)

**สิ่งที่ต้องทำ:**
1. **ฝั่ง Server (Flask):** เขียนโค้ดสร้าง API endpoint `/predict` ที่รับข้อมูล JSON เพื่อทำการทำนายกลุ่ม (Cluster)
2. **ฝั่ง Client (Requests):** เขียนฟังก์ชันจำลองการส่งข้อมูล (POST Request) ไปยัง Server ของเพื่อขอผลลัพธ์

*(หมายเหตุ: ใน Colab การรัน Flask อาจจะติด Block execution ให้เขียนโค้ดเพื่อแสดง Logic การทำงานเป็นหลัก หรือใช้ `werkzeug` / `threading` หากต้องการเทสรันจริง)*


### Server

In [7]:
# โค้ดฝั่ง Server (จำลองการทำงานของ Flask)
app = Flask(__name__)

# โหลด Model และ Scaler ที่บันทึกไว้
with open('best_penguin_model.pkl', 'rb') as f:
    saved_data = pk.load(f)
    deploy_model = saved_data['model']
    deploy_scaler = saved_data['scaler']

@app.route('/predict', methods=['POST'])
def predict_api():
    try:
        # 1. รับข้อมูล JSON
        data = request.get_json()

        # 2. แกะข้อมูลเรียงตามลำดับ Features ที่เทรน (culmen_length, culmen_depth, flipper_length, body_mass)
        features_list = [
            data["culmen_length_mm"],
            data["culmen_depth_mm"],
            data["flipper_length_mm"],
            data["body_mass_g"]
        ]

        # 3. นำข้อมูลใหม่ไป Scale ด้วยเงื่อนไขเดิม (สำคัญมาก)
        features_scaled = deploy_scaler.transform([features_list])

        # 4. ทำนายผล
        prediction = deploy_model.predict(features_scaled)

        # 5. ส่งผลลัพธ์กลับ
        return jsonify({'cluster_id': int(prediction[0])}), 200

    except Exception as e:
        return jsonify({'error': str(e)}), 400

# หมายเหตุ: ใน Colab การรัน app.run() จะทำให้เซลล์ค้าง
# หากต้องการรันเพื่อเทสต์จริงๆ ให้เอาเครื่องหมาย # ออก แต่ต้องระวังเซลล์ค้าง
# app.run(host='127.0.0.1', port=5000, debug=False)

print("Flask Server configuration loaded successfully (Logic mapped).")

Flask Server configuration loaded successfully (Logic mapped).


### Client

In [ ]:
def test_penguin_client(server_url):
    # ข้อมูลจำลองสำหรับทดสอบ (เพนกวิน 1 ตัว)
    mock_data = {
        "culmen_length_mm": 39.1,
        "culmen_depth_mm": 18.7,
        "flipper_length_mm": 181.0,
        "body_mass_g": 3750.0
    }

    print(f"Sending POST request to {server_url} ...")
    print(f"Data: {mock_data}")

    try:
        response = requests.post(
            url=server_url,
            headers={'Content-Type': 'application/json'},
            data=json.dumps(mock_data)
        )

        if response.status_code == 200:
            result = response.json()
            print(f"\n✅ Prediction Success! The penguin belongs to Cluster: {result['cluster_id']}")
            return result
        else:
            print(f"\n❌ Error {response.status_code}: {response.text}")

    except requests.exceptions.RequestException as e:
        print(f"\n❌ Connection Error (Server might not be running): {str(e)}")

# ทดสอบยิง Request (จะติด Connection Error หากไม่ได้รัน app.run() ด้านบน)
url = 'http://127.0.0.1:5000/predict'
test_penguin_client(url)